Evaluate the model and improve its accuracy

In [ ]:
""" create forecasts for the test period """

## split the prophet dataframe
train_size = int(len(prophet_df) * 0.8)
train = prophet_df[:train_size]
test = prophet_df[train_size:]

In [ ]:
## train only on training data
model_eval = Prophet(
    daily_seasonality = False,
    weekly_seasonality = False,
    yearly_seasonality = True,
    changepoint_prior_scale = 0.02
    ## started with 0.05, gradually reduced it to 0.02,
    ## to get the lowest possible MAE and RMSE
)
model_eval.fit(train)

In [ ]:
## forecast for the test period
future_test = model_eval.make_future_dataframe(periods=len(test))
forecast_test = model_eval.predict(future_test)

In [ ]:
""" align the predictions with actual values """

forecast_test = forecast_test.set_index("ds")
test = test.set_index("ds")

comparison = test.join(
    forecast_test[['yhat']],
    how = "left"
)

comparison.head()

In [ ]:
""" calculate the error metrics, MAE and RMSE """

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(comparison["y"], comparison["yhat"])
rmse = np.sqrt(mean_squared_error(comparison["y"], comparison["yhat"]))

print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

In [ ]:
""" visualise your prediction vs actual values """

plt.figure(figsize=(12, 6))
plt.plot(comparison.index, comparison["y"], label="Actual")
plt.plot(comparison.index, comparison["yhat"], label="Predicted")
plt.title("Actual vs Predicted Exchange Rate")
plt.legend()
plt.show()

In [ ]:
""" re-forecast next 30 days """

from typing_extensions import final
final_model = Prophet(
    daily_seasonality = False,
    weekly_seasonality = False,
    yearly_seasonality = True,
    changepoint_prior_scale = 0.1
)
final_model.fit(prophet_df)
future = final_model.make_future_dataframe(periods=30)
forecast = final_model.predict(future)
final_model.plot(forecast)